<p style="font-size: 20px; color:#66ffff; text-align: center">
     Importando as bibliotecas - 
     Importing the libraries
</p>


In [36]:
import pandas as pd

<p style="font-size: 20px; color:#00ff00; text-align: center">
   Importando e Tratando dados -
   Importing and Cleaning Data
</p>

In [37]:
def read_database(caminho_arquivo):
    excel_file = pd.ExcelFile(caminho_arquivo)
    planilhas = [nome for nome in excel_file.sheet_names if nome != "Parâmetros"]
    
    dfs = {}
    receita = 0
    for planilha in planilhas: 
        df = pd.read_excel(
            excel_file,
            sheet_name = planilha,
            skiprows=1,
            usecols="A:J",
            header=None
        )
        df = df.drop(columns=[7,8])
        df.columns= df.iloc[0]
        df = df[1:].reset_index(drop=True)

        if not df.empty:
            try:
                valor = df.iloc[0, 7] 
                valor_float = float(str(valor).replace(',', '.'))
                receita += valor_float
            except (ValueError, TypeError):
                ...
        chave = planilha.lower().replace(" ", "_").replace("2024", "24")
        dfs[chave] = df
        
    return dfs, receita



<p style="font-size: 20px; color:#66ffff; text-align: center">
   Normalizar os Valores monetários  -
   Standardize the monetary values
</p>

In [38]:
def normalizar_valor(valor):
    valor = str(valor).replace('R$', '').strip()

    if '.' in valor and ',' in valor:
        valor = valor.replace('.', '').replace(',', '.')
    elif ',' in valor:
        valor = valor.replace(',', '.')

    try:
        return float(valor)
    except ValueError:
        return 0.0




<p style="font-size: 20px; color:#66ffff; text-align: center">
   Limpezada de dados e Tipagem  -
   Data Cleaning and Typing
</p>

In [39]:
def limpar_dados(dfs):
    dfs_limpos = {}

    for chave, df in dfs.items():
        df = df.copy()
        df.columns = df.columns.astype(str).str.strip()

        if 'Valor' in df.columns:
            df['Valor'] = df['Valor'].apply(normalizar_valor).fillna(0)

        if 'Sazonalidade' in df.columns:
            df['Sazonalidade'] = pd.to_numeric(df['Sazonalidade'], errors='coerce').fillna(0).round(2)

        for coluna in ['Descrição', 'Data da Despesa']:
            if coluna in df.columns:
                df[coluna] = df[coluna].fillna('Desconhecido')

        if 'Data da Despesa' in df.columns:
            df['Data da Despesa'] = pd.to_datetime(df['Data da Despesa'], errors='coerce')
            df['mes_ano'] = df['Data da Despesa'].dt.to_period('M').astype(str)

        elif 'Vencimento' in df.columns:
            df['Vencimento'] = pd.to_datetime(df['Vencimento'], errors='coerce')
            df['mes_ano'] = df['Vencimento'].dt.to_period('M').astype(str)

        dfs_limpos[chave] = df

    return dfs_limpos


<p style="font-size: 20px; color:#66ffff; text-align: center">
   Consolidando Fluxo de Caixa para Predição  -
   Consolidating Cash Flow for Prediction
</p>

In [42]:
def consolidar_fluxo_de_caixa(dados):
    consolidado = []
    for chave, df in dados.items():
        if df.empty or not all(col in df.columns for col in ["Valor", "Tipo", "mes_ano"]):
            print(f"dataframe {chave} foi ignorado")
            continue

        df["Valor"] = pd.to_numeric(df["Valor"],errors="coerce")
        df["mes_ano"] = df["mes_ano"].astype(str)

        try:
            receita = float(df.columns[7])
        except:
            receita = 0 

        despesa_total = df[df["Tipo"] == "Despesa"]["Valor"].sum()
        lucro_bruto = receita - despesa_total

        mes_anos = df["mes_ano"].iloc[0]

        consolidado.append({
            "mes_ano": mes_anos, "receita_do_mes": receita, "despesa_total": despesa_total, "lucro_bruto": lucro_bruto 
        })
    df_consolidado = pd.DataFrame(consolidado).sort_values("mes_ano")


    df_consolidado["ds"] = pd.to_datetime(df_consolidado["mes_ano"] + "-01")
    df_consolidado = df_consolidado.sort_values("ds")
    df_consolidado["ano"] = df_consolidado["ds"].dt.year.replace(".", "")
    df_consolidado["mes"] = df_consolidado["ds"].dt.month

    df_consolidado["y"] = df_consolidado["lucro_bruto"]
    df_consolidado["variacao_mensal"] = df_consolidado["lucro_bruto"].pct_change().fillna(0).round(4)
    df_consolidado["despesas_acumulada"] = df_consolidado["despesa_total"].cumsum()
    df_consolidado["lucro_acumulado"] = df_consolidado["lucro_bruto"].cumsum()

    df_consolidado["margem_de_lucro"] = (
        (df_consolidado["lucro_bruto"] / df_consolidado["receita_do_mes"])
        .replace([float("inf"), -float("inf")], 0).round(4)
    )

    df_consolidado["alerta_risco"] = df_consolidado["lucro_bruto"].apply(
        lambda valor: "risco" if valor < 42000 else "ok"
    )

    colunas_formatar = [
    "receita_do_mes", "despesa_total", "lucro_bruto", 
    "despesas_acumulada", "lucro_acumulado", 'y'
    ]

    for col in colunas_formatar:
        df_consolidado[col] = df_consolidado[col].apply(
            lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        )

    return df_consolidado
        

<p style="font-size: 20px; color:#66ffff; text-align: center">
   Visualizando Arquivos - 
   Visualization Files 

</p>

In [43]:
caminho = r"C:\Users\admin\Documents\Project_Capstone\source\data\Project_Capstone.xlsx"
data, receita_total = read_database(caminho)

dados_limpos = limpar_dados(data)

df = consolidar_fluxo_de_caixa(dados_limpos)
df 

,mes_ano,receita_do_mes,despesa_total,lucro_bruto,ds,ano,mes,y,variacao_mensal,despesas_acumulada,lucro_acumulado,margem_de_lucro,alerta_risco
0,2024-09,"52.056,50","12.082,21","39.974,29",2024-09-01,2024,9,"39.974,29",0.0000,"12.082,21","39.974,29",0.7679,risco
1,2024-10,"64.547,00","12.067,73","52.479,27",2024-10-01,2024,10,"52.479,27",0.3128,"24.149,94","92.453,56",0.8130,ok
2,2024-11,"57.634,00","13.761,16","43.872,84",2024-11-01,2024,11,"43.872,84",-0.1640,"37.911,10","136.326,40",0.7612,ok
3,2024-12,"59.566,00","14.655,03","44.910,97",2024-12-01,2024,12,"44.910,97",0.0237,"52.566,13","181.237,37",0.7540,ok
4,2025-01,"55.351,00","13.577,81","41.773,19",2025-01-01,2025,1,"41.773,19",-0.0699,"66.143,94","223.010,56",0.7547,risco
5,2025-02,"54.901,00","14.508,86","40.392,14",2025-02-01,2025,2,"40.392,14",-0.0331,"80.652,80","263.402,70",0.7357,risco


<p style="font-size: 20px; color:#66ffff; text-align: center">
   Consolidação de Dataframes para Previsão
   Dataframe Consolidation Prection

</p>

In [ ]:
df.to_excel(r"C:\Users\admin\Documents\Project_Capstone\source\data\dados_consolidados.xlsx",index=False)
df.to_csv(r"C:\Users\admin\Documents\Project_Capstone\source\data\dados_consolidados.csv",index=False,sep=";")